# DRAS-5 Advanced Governance Workflows

This notebook extends the basic state-machine tutorial with timeout escalation, de-escalation requests, manual overrides, audit inspection, and comparative trajectory visualization.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from dras5 import DRAS5StateMachine, RiskState

## Timeout-driven escalation

In [ ]:
timeout_sm = DRAS5StateMachine(require_human_approval=False, session_id='timeout-demo')
timeout_sm.update(risk_score=0.55, t=0)
before = timeout_sm.current_state.name
timed_out = timeout_sm.check_timeout(t=130)
timeout_sm.auto_escalate(t=130)
after = timeout_sm.current_state.name
{'before': before, 'timed_out': timed_out, 'after': after}

## Controlled de-escalation comparison

In [ ]:
deesc_sm = DRAS5StateMachine(require_human_approval=False, session_id='deesc-demo')
deesc_sm.update(risk_score=0.75, t=0)
denied_state = deesc_sm.update(risk_score=0.15, t=40, deescalation_request=True, dual_approval=False)
approved_state = deesc_sm.update(
    risk_score=0.15,
    t=80,
    deescalation_request=True,
    dual_approval=True,
    rho_eff_series=[0.22, 0.20, 0.18, 0.17],
)
{'denied_state': denied_state.name, 'approved_state': approved_state.name}

## Emergency gate and manual override

In [ ]:
gate_sm = DRAS5StateMachine(require_human_approval=True, session_id='approval-demo')
gate_sm.update(risk_score=0.75, t=0)
blocked = gate_sm.update(risk_score=0.95, t=20, human_approved=False)
approved = gate_sm.update(risk_score=0.95, t=21, human_approved=True)
gate_sm.force_state(RiskState.ALERT, reason='operator_review', t=90)
{'blocked': blocked.name, 'approved': approved.name, 'post_override': gate_sm.current_state.name}

## Audit trail inspection

In [ ]:
history = gate_sm.get_history()
audit_df = pd.DataFrame([
    {
        'from_state': item.from_state.name,
        'to_state': item.to_state.name,
        'risk_score': item.risk_score,
        'trigger': item.trigger,
        'approved': item.approved,
        'session_id': item.session_id,
    }
    for item in history
])
audit_df

## Comparative workflow visualization

In [ ]:
state_order = {'SAFE': 1, 'MONITOR': 2, 'ALERT': 3, 'CRITICAL': 4, 'EMERGENCY': 5}
scenarios = {
    'Timeout path': [('SAFE', 0), ('ALERT', 0), ('CRITICAL', 130)],
    'Approved emergency': [('SAFE', 0), ('CRITICAL', 0), ('EMERGENCY', 21), ('ALERT', 90)],
    'Controlled de-escalation': [('SAFE', 0), ('CRITICAL', 0), ('CRITICAL', 40), ('ALERT', 80)],
}

fig, ax = plt.subplots(figsize=(10, 5))
for label, series in scenarios.items():
    times = [point[1] for point in series]
    states = [state_order[point[0]] for point in series]
    ax.plot(times, states, marker='o', linewidth=2, label=label)

ax.set_yticks([1, 2, 3, 4, 5], ['SAFE', 'MONITOR', 'ALERT', 'CRITICAL', 'EMERGENCY'])
ax.set_xlabel('Time (s)')
ax.set_ylabel('State')
ax.set_title('Advanced DRAS-5 workflow patterns')
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

## Export preview

In [ ]:
gate_sm.audit_log.to_json()[:400]